# Topic: Design a Real-Time Recommendation System

## Definition (30-second explanation)
*   **Analogy:** Finding a book in a massive library. 
    *   *Candidate Generation:* Asking the librarian to grab 100 books from the "Sci-Fi" and "History" aisles.
    *   *Ranking:* Sitting at a table and carefully scoring those 100 books based on the author, page count, and cover art.
    *   *Re-ranking:* Realizing you have 5 books by the exact same author, so you swap a few out for diversity before checking out.
*   A real-time recommendation system is a multi-stage funnel designed to filter millions of items down to a top-10 list in under 100 milliseconds.

## Why Interviewers Ask This
*   It is the most common ML system design question in the world. It tests your ability to balance heavy predictive accuracy (XGBoost/Deep Learning) with strict engineering latency constraints.

## Core Concepts (The 3-Stage Funnel)
1.  **Candidate Generation (Retrieval):**
    *   *Goal:* Filter millions of items down to ~500.
    *   *Method:* **Two-Tower Models**. One neural network creates an embedding for the User, and another creates an embedding for the Item. You store the item embeddings in a vector database (like ChromaDB) and do a fast Nearest-Neighbor search to find the 500 items closest to the user's embedding.
2.  **Ranking (Scoring):**
    *   *Goal:* Order those 500 items precisely.
    *   *Method:* Pass the 500 items and the user's detailed features into a heavy ML model (like **XGBoost** or a Deep Neural Network) to predict the exact probability of a click or purchase.
3.  **Re-ranking (Business Rules):**
    *   *Goal:* Final polish before the user sees it.
    *   *Method:* Apply lightweight filters. Remove items the user already bought, enforce diversity (so they don't see 10 identical videos in a row), and inject sponsored ads.

## The "Cold Start" Problem
*   **User Cold Start:** A brand new user signs up. You have zero historical data to feed into your Two-Tower or XGBoost model. 
    *   *Solution:* Serve a fallback "Popularity-based" feed (top trending items in their region) or ask them to select 3 topics during onboarding.
*   **Item Cold Start:** A brand new video is uploaded. It has zero views, so the models don't know who to show it to.
    *   *Solution:* Use Content-Based Filtering (extract text/tags from the video and show it to users who like those text tags) or use an Epsilon-Greedy algorithm to randomly inject the new video into a few feeds to gather initial data.

## Common Interview Traps
*   **The "One Giant Model" Trap:** Suggesting you will run XGBoost or a Deep Learning model on all 10 million items in the database at once. *Correction:* That will take 5 minutes to compute. You must use the 3-stage funnel to narrow it down to 500 items first.

## 45-Second Interview Answer
"To design a real-time recommendation system, I use a strict three-stage funnel to meet latency requirements. Stage one is Candidate Generation, where I use a Two-Tower model and a vector database to retrieve the top 500 most relevant items from a pool of millions. Stage two is Ranking, where I pass those 500 candidates into a heavier model, like XGBoost, to predict the exact click-through probability using dense user and item features. Finally, stage three is Re-ranking, where I apply business logic to remove duplicates, enforce diversity, and inject sponsored content before serving the final feed."

### Cheat Sheet: The Two-Tower Model Architecture

**The Core Question:** *Why not use the exact same model to generate embeddings for both users and items so they can be compared?*

**The Answer:** Because the input data shapes are fundamentally different. You cannot feed a user's click history and an article's raw text through the exact same neural network layers. 

**The Architecture**
You build two completely separate neural networks (the "Towers") standing side-by-side:
*   **The User Tower:** Ingests user features (e.g., age, location, last 50 clicked items, device type).
*   **The Item Tower:** Ingests item features (e.g., article text, author ID, category tags, publish date).

**The Magic (The Shared Vector Space)**
While the internal layers of the two towers are completely different, you engineer the **final output layer** of both towers to be the exact same size (e.g., a 256-dimensional vector). 

**How They Are Trained Together**
During training, you feed a User into Tower A and an Item they actually clicked into Tower B. 
1. The User Tower outputs a 256-D vector.
2. The Item Tower outputs a 256-D vector.
3. The Loss Function calculates the distance between them and tells both networks: *"Adjust your internal weights so that when this specific user and this specific item come through again, your final output vectors are pulled closer together."*

**The Analogy**
It is like having two different translation departments. One translates Japanese (User data) and the other translates French (Item data). However, they are both strictly trained to output their final reports in Esperanto. Because they both output in the exact same mathematical "language," you can run a lightning-fast cosine similarity search between them in your vector database.

## Practice Questions:

### Q1: The User Cold Start Problem
**Question:** 
A brand new user skips onboarding and lands on your platform. You have 100ms to show them 10 items, but your Two-Tower and Ranking models have zero historical data. Describe your fallback strategy and how you transition them to the personalized models.

**Answer:**
"For a total user cold start, I rely on a **Localized Popularity Baseline**. Even without explicit history, the HTTP request gives me metadata: IP address (region/country), browser language, and time of day. I use this to serve the top trending items for that specific cohort. 
Simultaneously, I implement an aggressive data-gathering strategy. I track **implicit feedback** (scroll depth, dwell time, mouse hovers) and prompt for lightweight **explicit feedback** (thumbs up/down UI widgets). By the next batch run, we have enough implicit interaction data to run algorithms like ALS (Alternating Least Squares) matrix factorization, allowing the user to transition into our standard personalized XGBoost ranking pipeline."

**Common Interview Traps:**
* Saying "I will just show them random items." Randomness causes immediate churn.
* Forgetting that metadata (Location, Time, Device) exists even if user history does not.